## STEP 0: Improt Training data 


In [38]:
from collections import defaultdict
import pandas as pd
import numpy as np
import pickle

In [39]:
df = pd.read_csv('split_cleaned_dataset.csv', sep=';')
print("First 5 rows:")
print("\nSplit distribution:")
print(df['split'].value_counts())

First 5 rows:

Split distribution:
split
train    345628
val       61539
test      59894
Name: count, dtype: int64


In [40]:
print(df[df['split'] == 'train'].shape)
print(df[df['split'] == 'val'].shape)

(345628, 23)
(61539, 23)


In [41]:
raw_train_val = df[df['split'].isin(['train', 'val'])].copy()

print(f"Total rows: {len(df)}")
print(f"Training rows: {len(raw_train_val)}")
print("\nTraining&Val data sample:")
print(f"\nColumns: {raw_train_val.columns.tolist()}")


Total rows: 467061
Training rows: 407167

Training&Val data sample:

Columns: ['Rank', 'Title', 'Artists', 'Date', 'Danceability', 'Energy', 'Loudness', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence', '# of Artist', 'Artist (Ind.)', '# of Nationality', 'Nationality', 'Continent', 'Points (Total)', 'Points (Ind for each Artist/Nat)', 'id', 'Song URL', 'Loudness_norm', 'first_appearance', 'split']


In [42]:
raw_train_val[raw_train_val['split'].isin(['val'])].head(5)

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm,first_appearance,split
57,67,Dandelions,Ruth B.,2023-05-29,0.609,0.692,-2.958,0.026,0.016,0.0,...,Nationality 1,Canada,Anglo-America,134,134.0,2eAvDnpXP5W0cVtiI0PUxV,https://open.spotify.com/track/2eAvDnpXP5W0cVt...,0.914199,2021-08-03,val
76,88,Set Fire to the Rain,Adele,2023-05-29,0.603,0.670,-3.882,0.025,0.004,0.0,...,Nationality 1,United Kingdom,Europe,113,113.0,7j4OmvkjRz0PrjFADlHfQx,https://open.spotify.com/track/7j4OmvkjRz0PrjF...,0.887397,2021-10-11,val
94,109,505,Arctic Monkeys,2023-05-29,0.520,0.852,-5.866,0.054,0.002,0.0,...,Nationality 1,United Kingdom,Europe,92,92.0,58ge6dfP91o9oXMzq3XkIS,https://open.spotify.com/track/58ge6dfP91o9oXM...,0.829848,2021-02-01,val
116,134,Ghost,Justin Bieber,2023-05-29,0.601,0.741,-5.569,0.048,0.185,0.0,...,Nationality 1,Canada,Anglo-America,67,67.0,6I3mqTwhRpn34SLVafSH7G,https://open.spotify.com/track/6I3mqTwhRpn34SL...,0.838463,2021-03-20,val
117,135,Can't Hold Us (feat. Ray Dalton),"Macklemore & Ryan Lewis, Macklemore, Ryan Lewi...",2023-05-29,0.633,0.927,-4.468,0.084,0.027,0.0,...,Nationality 1,United States,Anglo-America,66,16.5,22skzmqfdWrjJylampe0kt,https://open.spotify.com/track/22skzmqfdWrjJyl...,0.870399,2021-12-31,val


## STEP 1: Calculate Artist Features
Purpose: Get statistics for each unique artist (primary + collaborators)

1A: Extract All Artist Appearances

In [43]:
artist_appearances = defaultdict(list)

for idx, row in raw_train_val.iterrows():
    song_id = row['id']
    title = row['Title']
    artists_str = row['Artists']
    date = row['Date']
    rank = row['Rank']

    artists_list = [artist.strip() for artist in artists_str.split(',')]
    for position, artist_name in enumerate(artists_list):
        is_primary = (position == 0)
        
        appearance = {
            'song_id': song_id,
            'title': title,
            'date': date,
            'rank': rank,
            'is_primary': is_primary,
            'role': 'primary' if is_primary else 'collaborator',
            'artist_position': position
        }
        
        artist_appearances[artist_name].append(appearance)

print(artist_appearances['Ed Sheeran'][:2])  

all_appearances = []
for artist_name, appearances in artist_appearances.items():
    for appearance in appearances:
        all_appearances.append({
            'artist_name': artist_name,
            **appearance
        })

artist_appearances_df = pd.DataFrame(all_appearances)
artist_appearances_df['date'] = pd.to_datetime(artist_appearances_df['date'])

print(f"\nExtracted {len(artist_appearances_df)} artist appearances from {len(raw_train_val)} training rows")
print(f"Unique artists: {len(artist_appearances)}")
print("\nFirst 5 rows:")
print(artist_appearances_df.head())
print(f"\nData types:\n{artist_appearances_df.dtypes}")

[{'song_id': '0tgVpDi06FyKpA1z0VMD4v', 'title': 'Perfect', 'date': '2023-05-29', 'rank': 94, 'is_primary': True, 'role': 'primary', 'artist_position': 0}, {'song_id': '7qiZfU4dY1lWllzX7mPBI3', 'title': 'Shape of You', 'date': '2023-05-29', 'rank': 164, 'is_primary': True, 'role': 'primary', 'artist_position': 0}]

Extracted 568999 artist appearances from 407167 training rows
Unique artists: 1803

First 5 rows:
  artist_name                 song_id                             title  \
0       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
1       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
2       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
3       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   
4       Tyler  7KA4W4McWYRpgf0fWsJZWB  See You Again (feat. Kali Uchis)   

        date  rank  is_primary     role  artist_position  
0 2023-05-29    18        True  primary                0  
1 2023-05-28    1

In [44]:
artist_appearances_df['artist_name']

0         Tyler
1         Tyler
2         Tyler
3         Tyler
4         Tyler
          ...  
568994    Nause
568995    Nause
568996    Nause
568997    Nause
568998    Nause
Name: artist_name, Length: 568999, dtype: object

1B: Calculate Artist Statistics

In [45]:
primary_only = artist_appearances_df[artist_appearances_df['is_primary'] == True]
artist_stats = primary_only.groupby('artist_name').agg({
    'song_id': 'nunique',      
    'rank': ['mean', 'min']     
}).reset_index()


artist_stats.columns = ['artist_name', 'song_count', 'avg_rank', 'best_rank']


artist_stats['avg_rank'] = artist_stats['avg_rank'].round(2)


artist_stats = artist_stats.sort_values('song_count', ascending=False)

print(f"\nTotal unique artists: {len(artist_stats):,}")
print("\nArtist Statistics Summary:")
print(artist_stats.describe())

print(artist_stats.head(20).to_string(index=False))

top_performers = artist_stats.nsmallest(20, 'best_rank')
print(top_performers[['artist_name', 'song_count', 'avg_rank', 'best_rank']].to_string(index=False))

if 'Bad Bunny' in artist_stats['artist_name'].values:
    bb = artist_stats[artist_stats['artist_name'] == 'Bad Bunny'].iloc[0]
    print(f"\nBad Bunny:")
    print(f"  - Unique songs: {bb['song_count']}")
    print(f"  - Average rank: {bb['avg_rank']}")
    print(f"  - Best rank: {bb['best_rank']}")


song_count_dist = artist_stats['song_count'].value_counts().sort_index()
print("\nArtists by number of unique songs:")
for count, num_artists in song_count_dist.head(20).items():
    print(f"  {count} song(s): {num_artists:,} artists")

print(f"\nDataFrame shape: {artist_stats.shape}")
print(f"Columns: {artist_stats.columns.tolist()}")


Total unique artists: 1,258

Artist Statistics Summary:
        song_count     avg_rank    best_rank
count  1258.000000  1258.000000  1258.000000
mean      6.026232   133.371113    79.829094
std      12.553090    35.620093    59.325875
min       1.000000    31.200000     1.000000
25%       1.000000   105.070000    26.000000
50%       2.000000   134.265000    70.500000
75%       5.000000   161.500000   126.750000
max     174.000000   200.000000   200.000000
  artist_name  song_count  avg_rank  best_rank
 Taylor Swift         174    102.52          1
        Drake         138     96.15          1
          BTS         100     89.91          1
       Future          99    103.37          3
   Juice WRLD          98    100.35          2
       Eminem          79    127.91          1
Ariana Grande          78     87.39          1
  Post Malone          77     88.54          1
        Logic          76     95.71          2
   Ed Sheeran          72    104.27          1
 Trippie Redd        

1C: Convert to Dictionary (Lookup)

In [46]:
artist_stats_lookup = {}

for _, row in artist_stats.iterrows():
    artist_stats_lookup[row['artist_name']] = {
        'song_count': int(row['song_count']),
        'avg_rank': float(row['avg_rank']),
        'best_rank': int(row['best_rank'])
    }

print(f"\nTotal artists in lookup dictionary: {len(artist_stats_lookup):,}")

example_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']

for artist in example_artists:
    if artist in artist_stats_lookup:
        stats = artist_stats_lookup[artist]
        print(f"\n{artist}:")
        print(f"  - song_count: {stats['song_count']}")
        print(f"  - avg_rank: {stats['avg_rank']}")
        print(f"  - best_rank: {stats['best_rank']}")
    else:
        print(f"\n{artist}: Not found in dataset")



for i, (artist_name, stats) in enumerate(list(artist_stats_lookup.items())[:10], 1):
    print(f"\n{i}. '{artist_name}': {{")
    print(f"     'song_count': {stats['song_count']},")
    print(f"     'avg_rank': {stats['avg_rank']},")
    print(f"     'best_rank': {stats['best_rank']}")
    print(f"   }}")

test_artist = 'Ed Sheeran'
if test_artist in artist_stats_lookup:
    print(f"\nLookup result for '{test_artist}':")
    print(artist_stats_lookup[test_artist])
else:
    print(f"\n'{test_artist}' not found")

print(f"Type: {type(artist_stats_lookup)}")
print(f"Number of keys: {len(artist_stats_lookup)}")
print(f"Memory efficient: O(1) lookup time")


sample_artist = list(artist_stats_lookup.keys())[0]
sample_stats = artist_stats_lookup[sample_artist]
print("\nValue data types:")
print(f"  - song_count: {type(sample_stats['song_count']).__name__}")
print(f"  - avg_rank: {type(sample_stats['avg_rank']).__name__}")
print(f"  - best_rank: {type(sample_stats['best_rank']).__name__}")


Total artists in lookup dictionary: 1,258

Bad Bunny:
  - song_count: 66
  - avg_rank: 82.46
  - best_rank: 1

Peso Pluma: Not found in dataset

J Balvin:
  - song_count: 49
  - avg_rank: 95.42
  - best_rank: 1

1. 'Taylor Swift': {
     'song_count': 174,
     'avg_rank': 102.52,
     'best_rank': 1
   }

2. 'Drake': {
     'song_count': 138,
     'avg_rank': 96.15,
     'best_rank': 1
   }

3. 'BTS': {
     'song_count': 100,
     'avg_rank': 89.91,
     'best_rank': 1
   }

4. 'Future': {
     'song_count': 99,
     'avg_rank': 103.37,
     'best_rank': 3
   }

5. 'Juice WRLD': {
     'song_count': 98,
     'avg_rank': 100.35,
     'best_rank': 2
   }

6. 'Eminem': {
     'song_count': 79,
     'avg_rank': 127.91,
     'best_rank': 1
   }

7. 'Ariana Grande': {
     'song_count': 78,
     'avg_rank': 87.39,
     'best_rank': 1
   }

8. 'Post Malone': {
     'song_count': 77,
     'avg_rank': 88.54,
     'best_rank': 1
   }

9. 'Logic': {
     'song_count': 76,
     'avg_rank': 95.7

## STEP 2: Calculate Collaboration Features
Purpose: Get statistics about collaborators specifically

2A: Extract All Collaborators

In [47]:
collaborator_data = []

for idx, row in raw_train_val.iterrows():
    song_id = row['id']
    artists_str = row['Artists']
    date = row['Date']
    rank = row['Rank']
    
    
    artists_list = [artist.strip() for artist in artists_str.split(',')]
    

    primary_artist = artists_list[0]
    collaborators = artists_list[1:] 
    
   
    if len(collaborators) > 0:
        for collaborator_name in collaborators:
            collaborator_data.append({
                'collaborator': collaborator_name,
                'song_id': song_id,
                'rank': rank,
                'date': date
            })


collaborator_df = pd.DataFrame(collaborator_data)
collaborator_df['date'] = pd.to_datetime(collaborator_df['date'])

print(f"\nTotal collaborator appearances extracted: {len(collaborator_df):,}")
print(f"Unique collaborators: {collaborator_df['collaborator'].nunique():,}")
print(f"Unique songs with collaborators: {collaborator_df['song_id'].nunique():,}")


print(collaborator_df.head(20).to_string(index=False))


if 'Bad Bunny' in collaborator_df['collaborator'].values:
    bb_collab = collaborator_df[collaborator_df['collaborator'] == 'Bad Bunny'].head(10)
    print(bb_collab.to_string(index=False))
else:
    print("Bad Bunny not found as collaborator")


collab_counts = collaborator_df['collaborator'].value_counts().head(20)
for artist, count in collab_counts.items():
    print(f"  {artist}: {count:,} appearances")


print(f"Shape: {collaborator_df.shape}")
print(f"Columns: {collaborator_df.columns.tolist()}")
print(f"Date range: {collaborator_df['date'].min()} to {collaborator_df['date'].max()}")
print(f"Rank range: {collaborator_df['rank'].min()} to {collaborator_df['rank'].max()}")


Total collaborator appearances extracted: 161,832
Unique collaborators: 942
Unique songs with collaborators: 1,708
collaborator                song_id  rank       date
 The Creator 7KA4W4McWYRpgf0fWsJZWB    18 2023-05-29
  Kali Uchis 7KA4W4McWYRpgf0fWsJZWB    18 2023-05-29
   Daft Punk 7MXVkk9YMctZqd1Srtv4MB    43 2023-05-29
    Jay Rock 2HbKqm4o0w5wEeEFXm2sD4    63 2023-05-29
    Swae Lee 0RiRZpuVRbi7oqRdSMwhQY   100 2023-05-29
      Khalid 0u2P5u6lvoDfwTYjAADbn4   114 2023-05-29
  Macklemore 22skzmqfdWrjJylampe0kt   135 2023-05-29
  Ryan Lewis 22skzmqfdWrjJylampe0kt   135 2023-05-29
  Ray Dalton 22skzmqfdWrjJylampe0kt   135 2023-05-29
    Coldplay 6RUKPb4LETWmmr3iAEQktW   177 2023-05-29
      Wizkid 1zi7xx7UVEFkmKfv06H8x0   198 2023-05-29
        Kyla 1zi7xx7UVEFkmKfv06H8x0   198 2023-05-29
 The Creator 7KA4W4McWYRpgf0fWsJZWB    19 2023-05-28
  Kali Uchis 7KA4W4McWYRpgf0fWsJZWB    19 2023-05-28
   Daft Punk 7MXVkk9YMctZqd1Srtv4MB    44 2023-05-28
    Jay Rock 2HbKqm4o0w5wEeEFXm2sD4 

2B: Count Collaborator Frequencies

In [48]:
collaborator_df.shape

(161832, 4)

In [49]:
collaborator_frequency = collaborator_df.groupby('collaborator').size().reset_index(name='frequency')
collaborator_frequency = collaborator_frequency.sort_values('frequency', ascending=False)

print(f"\nTotal unique collaborators: {len(collaborator_frequency):,}")
print(f"Total collaborator appearances: {collaborator_frequency['frequency'].sum():,}")

print(collaborator_frequency.head(20).to_string(index=False))

example_collabs = ['Bad Bunny', 'J Balvin', 'Peso Pluma', 'Ozuna']

for collab in example_collabs:
    if collab in collaborator_frequency['collaborator'].values:
        freq = collaborator_frequency[collaborator_frequency['collaborator'] == collab].iloc[0]['frequency']
        print(f"\n{collab}:")
        print(f"  - Frequency: {freq:,} appearances as collaborator")
    else:
        print(f"\n{collab}: Not found as collaborator")

print(f"Mean frequency: {collaborator_frequency['frequency'].mean():.2f}")
print(f"Median frequency: {collaborator_frequency['frequency'].median():.0f}")
print(f"Min frequency: {collaborator_frequency['frequency'].min()}")
print(f"Max frequency: {collaborator_frequency['frequency'].max()}")

print(collaborator_frequency.tail(10).to_string(index=False))



Total unique collaborators: 942
Total collaborator appearances: 161,832
  collaborator  frequency
     Bad Bunny       5968
      J Balvin       5673
         Ozuna       3752
      Anuel AA       3028
        Khalid       2888
  Daddy Yankee       2812
      Swae Lee       2411
       Farruko       2354
      Dua Lipa       2285
 Lenny Tavárez       2135
   Myke Towers       1871
          Sech       1791
     Nicky Jam       1730
         Quavo       1664
 Justin Bieber       1531
     Daft Punk       1390
   Nicki Minaj       1353
        Darell       1338
      Coldplay       1301
Bradley Cooper       1288

Bad Bunny:
  - Frequency: 5,968 appearances as collaborator

J Balvin:
  - Frequency: 5,673 appearances as collaborator

Peso Pluma: Not found as collaborator

Ozuna:
  - Frequency: 3,752 appearances as collaborator
Mean frequency: 171.80
Median frequency: 34
Min frequency: 1
Max frequency: 5968
         collaborator  frequency
           Smokepurpp          1
            Jungl

2C: Identify Top 15 Collaborators

In [50]:
collaborator_frequency.head()

,collaborator,frequency
81,Bad Bunny,5968
354,J Balvin,5673
656,Ozuna,3752
57,Anuel AA,3028
437,Khalid,2888


In [51]:
top_collabs = collaborator_frequency.nlargest(15, 'frequency')

print("\nTop 15 Most Frequent Collaborators:")

print(top_collabs.to_string(index=False))


top_15_collabs_list = top_collabs['collaborator'].tolist()


print("Top 15 Collaborators List (for binary features):")

for i, collab in enumerate(top_15_collabs_list, 1):
    freq = top_collabs[top_collabs['collaborator'] == collab].iloc[0]['frequency']
    print(f"{i:2d}. {collab:<30} (Frequency: {freq:,})")


print(f"Total collaborators identified: {len(top_15_collabs_list)}")
print(f"Total appearances covered: {top_collabs['frequency'].sum():,}")
print(f"Percentage of all collaborator appearances: {(top_collabs['frequency'].sum() / collaborator_frequency['frequency'].sum() * 100):.2f}%")


print(f"Type: {type(top_15_collabs_list)}")
print(f"Length: {len(top_15_collabs_list)}")
print(f"\nFirst 5: {top_15_collabs_list[:5]}")
print(f"Last 5: {top_15_collabs_list[-5:]}")




Top 15 Most Frequent Collaborators:
 collaborator  frequency
    Bad Bunny       5968
     J Balvin       5673
        Ozuna       3752
     Anuel AA       3028
       Khalid       2888
 Daddy Yankee       2812
     Swae Lee       2411
      Farruko       2354
     Dua Lipa       2285
Lenny Tavárez       2135
  Myke Towers       1871
         Sech       1791
    Nicky Jam       1730
        Quavo       1664
Justin Bieber       1531
Top 15 Collaborators List (for binary features):
 1. Bad Bunny                      (Frequency: 5,968)
 2. J Balvin                       (Frequency: 5,673)
 3. Ozuna                          (Frequency: 3,752)
 4. Anuel AA                       (Frequency: 3,028)
 5. Khalid                         (Frequency: 2,888)
 6. Daddy Yankee                   (Frequency: 2,812)
 7. Swae Lee                       (Frequency: 2,411)
 8. Farruko                        (Frequency: 2,354)
 9. Dua Lipa                       (Frequency: 2,285)
10. Lenny Tavárez           

2D: Calculate Artist Popularity Scores

In [52]:
artist_appearances_df.head()

,artist_name,song_id,title,date,rank,is_primary,role,artist_position
0,Tyler,7KA4W4McWYRpgf0fWsJZWB,See You Again (feat. Kali Uchis),2023-05-29,18,True,primary,0
1,Tyler,7KA4W4McWYRpgf0fWsJZWB,See You Again (feat. Kali Uchis),2023-05-28,19,True,primary,0
2,Tyler,7KA4W4McWYRpgf0fWsJZWB,See You Again (feat. Kali Uchis),2023-05-27,24,True,primary,0
3,Tyler,7KA4W4McWYRpgf0fWsJZWB,See You Again (feat. Kali Uchis),2023-05-26,27,True,primary,0
4,Tyler,7KA4W4McWYRpgf0fWsJZWB,See You Again (feat. Kali Uchis),2023-05-25,20,True,primary,0


In [53]:
artist_appearances_df['popularity_score'] = 201 - artist_appearances_df['rank']

print("\nSample data with popularity scores (First 20 rows):")
print(artist_appearances_df[['artist_name', 'song_id', 'rank', 'popularity_score']].head(20).to_string(index=False))

artist_popularity = artist_appearances_df.groupby('artist_name')['popularity_score'].mean().reset_index()
artist_popularity.columns = ['artist_name', 'avg_popularity']

artist_popularity['avg_popularity'] = artist_popularity['avg_popularity'].round(2)

artist_popularity = artist_popularity.sort_values('avg_popularity', ascending=False)


print(f"Total artists: {len(artist_popularity):,}")
print(f"\nPopularity score distribution:")
print(artist_popularity['avg_popularity'].describe())


print(artist_popularity.head(20).to_string(index=False))



example_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']

for artist in example_artists:
    if artist in artist_popularity['artist_name'].values:
        pop = artist_popularity[artist_popularity['artist_name'] == artist].iloc[0]['avg_popularity']
        print(f"\n{artist}:")
        print(f"  - Average popularity score: {pop}")
    else:
        print(f"\n{artist}: Not found in dataset")

artist_popularity_lookup = dict(zip(artist_popularity['artist_name'], artist_popularity['avg_popularity']))

print(f"Total artists in lookup dictionary: {len(artist_popularity_lookup):,}")
print(f"Type: {type(artist_popularity_lookup)}")


for i, (artist, pop) in enumerate(list(artist_popularity_lookup.items())[:10], 1):
    print(f"{i:2d}. '{artist}': {pop}")


test_artists = ['Bad Bunny', 'Peso Pluma', 'J Balvin']
for artist in test_artists:
    if artist in artist_popularity_lookup:
        print(f"{artist}: {artist_popularity_lookup[artist]}")
    else:
        print(f"{artist}: Not found")



Sample data with popularity scores (First 20 rows):
artist_name                song_id  rank  popularity_score
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    19               182
      Tyler 7KA4W4McWYRpgf0fWsJZWB    24               177
      Tyler 7KA4W4McWYRpgf0fWsJZWB    27               174
      Tyler 7KA4W4McWYRpgf0fWsJZWB    20               181
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    17               184
      Tyler 7KA4W4McWYRpgf0fWsJZWB    14               187
      Tyler 7KA4W4McWYRpgf0fWsJZWB    16               185
      Tyler 7KA4W4McWYRpgf0fWsJZWB    24               177
      Tyler 7KA4W4McWYRpgf0fWsJZWB    23               178
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    18               183
      Tyler 7KA4W4McWYRpgf0fWsJZWB    15               186
      Tyler 7KA4W4McWYRpgf0fWsJZWB    13               188
   

2E: Calculate Global Mean Popularity

In [54]:
global_mean_popularity = artist_popularity['avg_popularity'].mean()

print(f"\nGlobal Mean Popularity: {global_mean_popularity:.2f}")


print(f"Total artists in calculation: {len(artist_popularity):,}")
print(f"Min artist popularity: {artist_popularity['avg_popularity'].min():.2f}")
print(f"Max artist popularity: {artist_popularity['avg_popularity'].max():.2f}")
print(f"Median artist popularity: {artist_popularity['avg_popularity'].median():.2f}")
print(f"Global mean popularity: {global_mean_popularity:.2f}")




Global Mean Popularity: 69.23
Total artists in calculation: 1,803
Min artist popularity: 1.00
Max artist popularity: 191.59
Median artist popularity: 68.43
Global mean popularity: 69.23


## STEP 3: Save all the lookup dictionaries and lists for artists and Pickle file
Purpose: Get statistics about collaborators specifically

In [55]:

features_to_save = {
    'artist_stats_lookup_T': artist_stats_lookup,
    'top_15_collabs_list_T': top_15_collabs_list,
    'artist_popularity_lookup_T': artist_popularity_lookup,
    'global_mean_popularity_T': global_mean_popularity
}

with open('artist_features_fortest.pkl', 'wb') as f:
    pickle.dump(features_to_save, f)

print("All features saved to 'artist_features.pkl'")
print(f"File contains: {list(features_to_save.keys())}")

All features saved to 'artist_features.pkl'
File contains: ['artist_stats_lookup_T', 'top_15_collabs_list_T', 'artist_popularity_lookup_T', 'global_mean_popularity_T']
